<a href="https://colab.research.google.com/github/yernarbakatay/Human-vs-AI-generated-Text-Detection/blob/main/RoBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing the model


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("/content/processed_dataset.csv")
data.head()

ParserError: Error tokenizing data. C error: EOF inside string starting at row 5494

In [ ]:
data.columns

In [ ]:
df = data[["text100", "ai_target"]].rename(columns={"text100":"text", "ai_target": "label"}) # gemini says robert is picky and it usually it expects to see these names for columns
df.isnull().sum()

In [ ]:
df = df.dropna()

df.isnull().sum()

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

NameError: name 'df' is not defined

In [ ]:
# load the model ROBERT
from sklearn.model_selection import train_test_split

# 1. Split into Train and Temp (80% / 20%)
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

# 2. Split Temp into Validation and Test (10% / 10%)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")


## convert to Hugging Face format
### apparently, HF is also picky

In [ ]:
from datasets import Dataset, DatasetDict

# Convert the DataFrames into Hugging Face Dataset objects
hg_train = Dataset.from_pandas(train_df)
hg_val = Dataset.from_pandas(val_df)
hg_test = Dataset.from_pandas(test_df)

# Group them into a single DatasetDict for easy management
my_dataset = DatasetDict({
    'train': hg_train,
    'validation': hg_val,
    'test': hg_test
})

# Pandas sometimes leaves behind an index column during conversion.
# Let's clean that up if it exists:
if "__index_level_0__" in my_dataset["train"].column_names:
    my_dataset = my_dataset.remove_columns(["__index_level_0__"])

print(my_dataset)

## Tokenize

In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized_ds = my_dataset.map(tokenize_fn, batched=True)

## LoRA is HERE! + Freezing

In [ ]:
## INITIALIZE AND TRAIN
from transformers import TrainingArguments, Trainer, RobertaForSequenceClassification
# from peft import LoraConfig, get_peft_model, TaskType  <-- COMMENT THIS OUT

# Initialize the model
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

# --- START OF LORA CODE TO COMMENT OUT ---
# Set up LoRA Configuration
# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,
#     r=8,
#     lora_alpha=16,
#     target_modules=["query", "value"],
#     lora_dropout=0.1
# )

# Wrap the base model with peft
# model = get_peft_model(model, lora_config)

# Print the percentage of trainable parameters
# model.print_trainable_parameters()
# --- END OF LORA CODE TO COMMENT OUT ---


# --- START OF NEW FREEZING CODE ---
# 1. Freeze the embedding layer (maps raw text to vector space)
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False

# 2. Freeze the first 8 layers (0 through 7)
for layer in model.roberta.encoder.layer[:8]:
    for param in layer.parameters():
        param.requires_grad = False

# Verify the reduction in parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
# --- END OF NEW FREEZING CODE ---


training_args = TrainingArguments(
    output_dir='./roberta_ai_detector',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir='./logs',
    fp16=True,
)

# Bring it all together into the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
)

# Start the actual fine-tuning!
trainer.train()

In [ ]:
# Run evaluation on the test dataset
test_results = trainer.evaluate(tokenized_ds['test'])
print(test_results)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Get predictions
predictions = trainer.predict(tokenized_ds['test'])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

In [ ]:
# 2. Create the matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI'])

# 3. Plot it
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix: AI vs Human Detector')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))

## train the data using new text


In [ ]:
# Saves the model and tokenizer to a folder
trainer.save_model("./freezing_ai_detector_model")
tokenizer.save_pretrained("./freezing_ai_detector_model")